# 04 — Modeling, Hyperparameter Optimization & Clinical-Safety Ensemble

**Project:** End-to-End Heart Disease Prediction System
**Stage:** 4 / 5 — Modeling & Ensemble

## Objective
This is the modeling core of the project. It progresses through four phases:

1. **Baseline evaluation** — six algorithm families under stratified 5-fold CV.
2. **Hyperparameter optimization** — Optuna (Tree-structured Parzen Estimator)
   tuning for XGBoost, LightGBM, and CatBoost.
3. **Ensembling** — comparing Soft Voting vs Stacking of the tuned base learners.
4. **Dual-threshold decision policy** — the centerpiece of the clinical-safety
   framing. A single global accuracy/F1 score hides an important asymmetry:
   in a screening context, a **False Negative** (telling a sick patient they
   are healthy) is far more costly than a **False Positive** (an extra
   follow-up test for a healthy patient). We therefore calibrate *two*
   decision thresholds on the same trained ensemble:
   - a **Leaderboard threshold**, which maximizes F1-Macro,
   - a **Clinical-Safety threshold**, which is tuned so that recall on the
     positive (disease) class reaches at least 95% — explicitly minimizing
     missed diagnoses, even at the cost of more false alarms.

## Input
- `data/processed/train_clean.csv`
- `data/processed/test_clean.csv`

## Output
- Tuned base-learner artifacts → `models/`
- Two deployment-ready ensemble variants (`model_leaderboard.pkl`,
  `model_clinical_safety.pkl`) → `models/`
- `reports/figures/` comparison charts
- `reports/submission/submission.csv`

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import copy
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import f1_score, confusion_matrix, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.base import BaseEstimator, ClassifierMixin
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

PROCESSED_DIR = "../data/processed"
MODELS_DIR    = "../models"
FIGURES_DIR   = "../reports/figures"
SUBMISSION_DIR = "../reports/submission"

N_TRIALS = 50   # Optuna trials per model — reduce for a quick local run
RANDOM_STATE = 42

## 2. Load Processed Data

In [ ]:
train = pd.read_csv(f"{PROCESSED_DIR}/train_clean.csv")
test  = pd.read_csv(f"{PROCESSED_DIR}/test_clean.csv")

X = train.drop(columns=['HeartDisease'])
y = train['HeartDisease']

print(f"Train feature matrix : {X.shape}")
print(f"Test feature matrix  : {test.shape}")
print(f"Target distribution  :\n{y.value_counts(normalize=True).round(3)}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 3. Baseline Models — 5-Fold Stratified Cross-Validation

Establishing baselines first is what makes the later Optuna improvement
*measurable* rather than anecdotal. F1-Macro is used as the primary metric
since it weighs both classes equally regardless of class balance.

In [ ]:
models_baseline = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'Gradient Boosting'  : GradientBoostingClassifier(random_state=RANDOM_STATE),
    'XGBoost'            : XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0),
    'LightGBM'           : LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
    'CatBoost'           : CatBoostClassifier(verbose=0, random_state=RANDOM_STATE),
}

baseline_scores = {}
for name, model in models_baseline.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='f1_macro', n_jobs=1)
    baseline_scores[name] = scores.mean()
    print(f"{name:<22}: F1-Macro = {scores.mean():.4f} +/- {scores.std():.4f}")

## 4. Hyperparameter Optimization with Optuna

Optuna's Tree-structured Parzen Estimator (TPE) sampler is used to search the
hyperparameter space of the three gradient-boosting families more efficiently
than grid search. Each trial is scored with the same 5-fold CV used for the
baseline, so improvements are directly comparable.

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 300, 800),
        'max_depth'       : trial.suggest_int('max_depth', 3, 6),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 5),
        'gamma'           : trial.suggest_float('gamma', 0, 0.3),
        'eval_metric': 'logloss', 'random_state': RANDOM_STATE, 'verbosity': 0, 'n_jobs': 1,
    }
    return cross_val_score(
        XGBClassifier(**params), X, y, cv=cv, scoring='f1_macro', n_jobs=1
    ).mean()


def objective_lgbm(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 300, 800),
        'max_depth'        : trial.suggest_int('max_depth', 3, 6),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 15, 63),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 40),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 0, 0.5),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 0, 0.5),
        'random_state': RANDOM_STATE, 'verbose': -1, 'n_jobs': 1,
    }
    return cross_val_score(
        LGBMClassifier(**params), X, y, cv=cv, scoring='f1_macro', n_jobs=1
    ).mean()


def objective_cat(trial):
    params = {
        'iterations'   : trial.suggest_int('iterations', 300, 800),
        'depth'        : trial.suggest_int('depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'l2_leaf_reg'  : trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'border_count' : trial.suggest_int('border_count', 32, 128),
        'verbose': 0, 'random_state': RANDOM_STATE,
    }
    return cross_val_score(
        CatBoostClassifier(**params), X, y, cv=cv, scoring='f1_macro', n_jobs=1
    ).mean()

In [ ]:
print("Tuning XGBoost...")
study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)

print("Tuning LightGBM...")
study_lgbm = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS)

print("Tuning CatBoost...")
study_cat = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study_cat.optimize(objective_cat, n_trials=N_TRIALS)

print(f"\nBest XGBoost  F1: {study_xgb.best_value:.4f}")
print(f"Best LightGBM F1: {study_lgbm.best_value:.4f}")
print(f"Best CatBoost F1: {study_cat.best_value:.4f}")

In [ ]:
best_xgb  = XGBClassifier(**study_xgb.best_params,
                           eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0, n_jobs=1)
best_lgbm = LGBMClassifier(**study_lgbm.best_params,
                            random_state=RANDOM_STATE, verbose=-1, n_jobs=1)
best_cat  = CatBoostClassifier(**study_cat.best_params,
                                verbose=0, random_state=RANDOM_STATE)
best_rf   = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=1)

tuned_models = {
    'XGBoost'      : best_xgb,
    'LightGBM'     : best_lgbm,
    'CatBoost'     : best_cat,
    'Random Forest': best_rf,
}

## 5. Baseline vs Tuned — Quantifying the Optimization Gain

In [ ]:
tuned_scores = {}
oof_preds    = {}

for name, model in tuned_models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='f1_macro', n_jobs=1)
    tuned_scores[name] = scores.mean()
    oof_preds[name]    = cross_val_predict(model, X, y, cv=cv, n_jobs=1)

print(f"\n{'Model':<22} {'Baseline':>10} {'Tuned':>10} {'Delta':>8}")
print("-" * 52)
for name in tuned_models:
    base  = baseline_scores.get(name, 0)
    tuned = tuned_scores[name]
    delta = tuned - base
    arrow = "UP" if delta > 0 else "DOWN"
    print(f"{name:<22} {base:>10.4f} {tuned:>10.4f} {arrow}{abs(delta):>6.4f}")

In [ ]:
models_list = list(tuned_models.keys())
baseline_vals = [baseline_scores[m] for m in models_list]
tuned_vals    = [tuned_scores[m] for m in models_list]
deltas        = [t - b for t, b in zip(tuned_vals, baseline_vals)]

y_pos = np.arange(len(models_list))
height = 0.35

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
color_baseline = '#A9CCE3'
color_tuned    = '#154360'

rects1 = ax.barh(y_pos - height/2, baseline_vals, height, label='Baseline', color=color_baseline)
rects2 = ax.barh(y_pos + height/2, tuned_vals, height, label='Tuned (Optuna)', color=color_tuned)

ax.set_xlabel('F1-Macro Score', fontsize=12, fontweight='bold')
ax.set_title('Model Performance: Baseline vs Optuna-Tuned', fontsize=15, fontweight='bold', pad=15)
ax.set_yticks(y_pos)
ax.set_yticklabels(models_list, fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.legend(loc='upper left', bbox_to_anchor=(0, -0.08), ncol=2, frameon=False)
ax.grid(axis='x', linestyle='--', alpha=0.5)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

for rects, vals, is_tuned in [(rects1, baseline_vals, False), (rects2, tuned_vals, True)]:
    for rect, v in zip(rects, vals):
        ax.annotate(f'{v:.4f}', xy=(v, rect.get_y() + rect.get_height()/2),
                     xytext=(6, 0), textcoords='offset points',
                     ha='left', va='center', fontsize=11,
                     fontweight='bold' if is_tuned else 'normal',
                     color=color_tuned if is_tuned else '#555555')

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/baseline_vs_tuned.png", dpi=150, transparent=True, bbox_inches='tight')
plt.show()

## 6. Out-of-Fold Confusion Matrices — Per Base Learner

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, preds) in zip(axes, oof_preds.items()):
    cm = confusion_matrix(y, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    f1 = f1_score(y, preds, average='macro')
    ax.set_title(f'{name}\nF1-Macro: {f1:.4f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrix — Each Tuned Base Learner (Out-of-Fold)', y=1.05)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/oof_confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Ensembling — Soft Voting vs Stacking

Both ensemble strategies combine the four tuned base learners. Soft Voting
averages predicted probabilities; Stacking trains a meta-learner (Logistic
Regression) on out-of-fold predictions of the base learners. We compare both
under the same CV protocol and keep whichever generalizes better.

In [ ]:
estimators_list = [
    ('xgb',  best_xgb),
    ('lgbm', best_lgbm),
    ('cat',  best_cat),
    ('rf',   best_rf),
]

voting_ensemble = VotingClassifier(estimators=estimators_list, voting='soft', n_jobs=1)

stacking_ensemble = StackingClassifier(
    estimators=estimators_list,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=cv, n_jobs=1,
    passthrough=False
)

ensemble_scores = {}
for name, ensemble in [('Soft Voting', voting_ensemble), ('Stacking', stacking_ensemble)]:
    scores = cross_val_score(ensemble, X, y, cv=cv, scoring='f1_macro', n_jobs=1)
    ensemble_scores[name] = scores.mean()
    print(f"{name:<15}: F1-Macro = {scores.mean():.4f} +/- {scores.std():.4f}")

In [ ]:
# Select the ensemble strategy with the higher CV F1-Macro
best_ensemble_name = max(ensemble_scores, key=ensemble_scores.get)
final_ensemble = voting_ensemble if best_ensemble_name == 'Soft Voting' else stacking_ensemble
print(f"Selected ensemble strategy: {best_ensemble_name}")

print("Fitting final ensemble on the full training set...")
final_ensemble.fit(X, y)

In [ ]:
joblib.dump(best_xgb,  f"{MODELS_DIR}/best_xgb.pkl")
joblib.dump(best_lgbm, f"{MODELS_DIR}/best_lgbm.pkl")
joblib.dump(best_cat,  f"{MODELS_DIR}/best_cat.pkl")
joblib.dump(final_ensemble, f"{MODELS_DIR}/final_ensemble.pkl")
print("Base learners and final ensemble saved to models/.")

## 8. Dual-Threshold Analysis — Leaderboard Score vs Clinical Safety

This is the project's key clinical-safety contribution. A single threshold of
0.5 is a statistical convention, not a clinical one. We instead calibrate two
decision policies from the same trained ensemble's out-of-fold probabilities:

- **Leaderboard threshold** — the value of *t* that **maximizes F1-Macro**
  across a fine grid. This is the threshold a model would use if optimizing
  purely for a competition leaderboard metric.
- **Clinical-Safety threshold** — the **lowest** value of *t* (searched from
  a conservative grid) at which recall on the positive (disease) class is
  **>= 95%**. Lowering the decision threshold makes the model flag more
  patients as "at risk," which trades additional false positives (follow-up
  tests for healthy patients) for far fewer false negatives (missed
  diagnoses) — the right trade-off for a screening tool.

In [ ]:
y_proba_final = cross_val_predict(
    final_ensemble, X, y, cv=cv, n_jobs=1, method='predict_proba'
)[:, 1]

# --- Leaderboard strategy: maximize F1-Macro ---
thresh_leaderboard = 0.5
best_f1 = 0
for t in np.linspace(0.3, 0.7, 100):
    preds = (y_proba_final >= t).astype(int)
    f1 = f1_score(y, preds, average='macro')
    if f1 > best_f1:
        best_f1 = f1
        thresh_leaderboard = t

# --- Clinical safety strategy: recall (class 1) >= 95% ---
thresh_clinical = 0.35
for t in np.linspace(0.2, 0.5, 100):
    preds = (y_proba_final >= t).astype(int)
    if recall_score(y, preds) >= 0.95:
        thresh_clinical = t

y_pred_leaderboard = (y_proba_final >= thresh_leaderboard).astype(int)
y_pred_clinical    = (y_proba_final >= thresh_clinical).astype(int)

f1_leaderboard = f1_score(y, y_pred_leaderboard, average='macro')
f1_clinical    = f1_score(y, y_pred_clinical, average='macro')
recall_leaderboard = recall_score(y, y_pred_leaderboard)
recall_clinical     = recall_score(y, y_pred_clinical)

cm_lead = confusion_matrix(y, y_pred_leaderboard)
cm_clin = confusion_matrix(y, y_pred_clinical)
fn_leaderboard = cm_lead[1, 0]
fn_clinical    = cm_clin[1, 0]

print(f"Leaderboard threshold : {thresh_leaderboard:.3f} | F1-Macro: {f1_leaderboard:.4f} | Recall: {recall_leaderboard:.4f} | False Negatives: {fn_leaderboard}")
print(f"Clinical   threshold  : {thresh_clinical:.3f} | F1-Macro: {f1_clinical:.4f} | Recall: {recall_clinical:.4f} | False Negatives: {fn_clinical}")
print(f"\n>>> Switching from the Leaderboard threshold to the Clinical-Safety threshold reduces missed-diagnosis (False Negative) cases from {fn_leaderboard} to {fn_clinical}.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_lead, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Pred: Healthy', 'Pred: Disease'],
            yticklabels=['True: Healthy', 'True: Disease'])
axes[0].set_title(f'Leaderboard Strategy (Max F1)\nThreshold: {thresh_leaderboard:.3f} | F1-Macro: {f1_leaderboard:.4f}')

sns.heatmap(cm_clin, annot=True, fmt='d', cmap='Reds', ax=axes[1],
            xticklabels=['Pred: Healthy', 'Pred: Disease'],
            yticklabels=['True: Healthy', 'True: Disease'])
axes[1].set_title(f'Clinical-Safety Strategy (Recall >= 95%)\nThreshold: {thresh_clinical:.3f} | F1-Macro: {f1_clinical:.4f}')

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/dual_threshold_confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Packaging Deployment-Ready Model Variants

`ThresholdWrapper` bundles the fitted ensemble together with its decision
threshold into a single scikit-learn-compatible estimator, so each policy can
be deployed, versioned, and swapped independently without re-deriving the
threshold from scratch.

In [ ]:
class ThresholdWrapper(BaseEstimator, ClassifierMixin):
    """Wraps a fitted probabilistic model together with a decision threshold."""
    def __init__(self, model, threshold):
        self.model     = model
        self.threshold = threshold
        self.classes_  = np.array([0, 1])

    def fit(self, X, y=None):
        return self  # model is already fitted upstream

    def predict(self, X):
        proba = self.model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.model.predict_proba(X)


model_leaderboard = ThresholdWrapper(final_ensemble, thresh_leaderboard)
model_clinical    = ThresholdWrapper(copy.deepcopy(final_ensemble), thresh_clinical)

joblib.dump(model_leaderboard, f"{MODELS_DIR}/model_leaderboard.pkl")
joblib.dump(model_clinical,    f"{MODELS_DIR}/model_clinical_safety.pkl")

print("Two deployment-ready model variants saved:")
print(f"  -> {MODELS_DIR}/model_leaderboard.pkl")
print(f"  -> {MODELS_DIR}/model_clinical_safety.pkl")

## 10. Generate Test-Set Predictions / Submission File

In [ ]:
y_pred_test = model_leaderboard.predict(test)

submission = pd.DataFrame({
    'id'          : pd.read_csv(f"{PROCESSED_DIR}/../raw/test.csv", sep=';')['id'],
    'HeartDisease': y_pred_test
})
submission.to_csv(f"{SUBMISSION_DIR}/submission.csv", index=False)

print(f"Submission exported: {SUBMISSION_DIR}/submission.csv")
print(f"Predicted Healthy (0): {(y_pred_test == 0).sum()}")
print(f"Predicted Disease (1): {(y_pred_test == 1).sum()}")

---
**Next notebook:** `05_Explainable_AI_Reasoning.ipynb` — using a local LLM to
turn the highest-confidence positive predictions into human-readable clinical
reasoning.